# RouteAlert — เทรน Accident / Non-Accident Classifier

เทรนจริงด้วย Transfer Learning จาก MobileNetV2 (ไม่ใช่ synthetic data) แล้ว export เป็น TFLite
ให้ตรงกับ `lib/core/ml/accident_image_classifier_service.dart` ในแอป Flutter พอดี

**ก่อนรัน:** เตรียมรูปภาพ 2 โฟลเดอร์ `accident/` กับ `non_accident/` บีบอัดเป็น `dataset.zip`
(อย่างน้อยโฟลเดอร์ละ 50-100 รูป ยิ่งเยอะยิ่งแม่น)

In [ ]:
# ==== 1. เตรียมสภาพแวดล้อม ====
!pip install -q tensorflow

In [ ]:
# ==== 2. อัปโหลด dataset.zip ====
# โครงสร้างที่ต้องมีในไฟล์ zip:
#   dataset/
#     accident/       (รูปอุบัติเหตุ เช่น รถชน, รถคว่ำ, มอเตอร์ไซค์ล้ม)
#     non_accident/   (รูปทั่วไป เช่น เซลฟี่, ห้อง, ถนนปกติ, ภาพมืด/เบลอ)
from google.colab import files
uploaded = files.upload()  # เลือก dataset.zip ที่เตรียมไว้

import zipfile
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')

In [ ]:
# ==== 3. โหลด dataset ====
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print('Classes (เรียงตามตัวอักษร):', class_names)
# ต้องได้ ['accident', 'non_accident'] — ลำดับนี้จะตรงกับ accident_labels.txt ที่ export ออกไป

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(200).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# ==== 4. สร้างโมเดล (Transfer Learning จาก MobileNetV2 ที่เทรนบน ImageNet มาแล้ว) ====
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # freeze base ก่อน (transfer learning ขั้นที่ 1)

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
# ==== 5. เทรน ====
EPOCHS = 15
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

# ถ้า accuracy ยังไม่พอใจ ลอง fine-tune ขั้นที่ 2: ปลดล็อก base_model บางชั้น
# แล้วเทรนต่อด้วย learning rate ต่ำมากๆ (เช่น 1e-5) อีกสัก 5-10 epoch

In [ ]:
# ==== 6. ตรวจผลลัพธ์คร่าวๆ ก่อน export ====
loss, acc = model.evaluate(val_ds)
print(f'Validation accuracy: {acc*100:.1f}%')

In [ ]:
# ==== 7. Export เป็น TFLite (ให้ตรงกับที่ accident_image_classifier_service.dart คาดไว้) ====
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('accident_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

with open('accident_labels.txt', 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print('Export เสร็จแล้ว: accident_classifier.tflite, accident_labels.txt')

In [ ]:
# ==== 8. ดาวน์โหลดไฟล์ ====
# นำไฟล์ทั้งสองไปวางที่ assets/models/ ในโปรเจกต์ Flutter แล้วรันแอปได้เลย
# ไม่ต้องแก้โค้ด Dart เพิ่ม (เตรียม path/pubspec ไว้ให้แล้ว)
files.download('accident_classifier.tflite')
files.download('accident_labels.txt')